In [1]:
import sqlite3
import pandas as pd

# 1. db 파일 연결
db_path = 'login_log.db'
conn = sqlite3.connect(db_path)

# 2. DB 안에 있는 모든 테이블 이름 확인하기
query_table = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(query_table, conn)
print("이 DB에 있는 테이블 목록:\n", tables)

conn.close()

이 DB에 있는 테이블 목록:
             name
0   test_dataset
1  train_dataset


In [2]:
import sqlite3
import pandas as pd

db_path = 'login_log.db'
conn = sqlite3.connect(db_path)

# train_dataset의 전체 데이터 구조(상위 5줄) 가져오기
query_preview = "SELECT * FROM train_dataset LIMIT 5;"
df_preview = pd.read_sql_query(query_preview, conn)

print("=== 학습 데이터 미리보기 ===")
# 주피터 노트북 환경에서는 print()보다 display()를 쓰면 표를 엑셀처럼 예쁘게 그려줘!
display(df_preview) 

conn.close()

=== 학습 데이터 미리보기 ===


,timestamp,user_id,ip_address,location,device_type,os_type,browser,success,session_duration,target
0,2025-07-23 0:10,6a214818-e78e-47fc-a728-226c70b18e37,148.6.34.231,"Lake Robertbury, Bhutan",Tablet,Windows,Edge,FALSE,3167,TRUE
1,2025-12-09 12:43,6ad7f4f8-3b0b-45c8-b6aa-2cf6a10b2040,134.106.53.46,"Jeffersonville, India",PC,MacOS,Edge,TRUE,3360,FALSE
2,2025-04-10 4:15,ec4aa120-d196-45af-98ca-4a13ec08d354,164.233.90.226,"Michaelhaven, French Polynesia",PC,Linux,Firefox,TRUE,1836,TRUE
3,2026-02-13 2:22,71041fa0-b11c-4df8-ace3-59a60b13d613,205.62.126.123,"Port Seth, Gambia",PC,Windows,Chrome,TRUE,46,FALSE
4,2025-02-15 12:41,130c32b7-28e7-4d6c-99e0-2e17a439550e,157.225.10.229,"Georgeview, Central African Republic",PC,iOS,Safari,TRUE,2039,TRUE


In [3]:
import sqlite3
import pandas as pd

db_path = 'login_log.db'
conn = sqlite3.connect(db_path)

# success 컬럼이 'FALSE' 이거나 0인 실패 기록만 추출하는 쿼리
query_fail = "SELECT * FROM train_dataset WHERE success = 'FALSE' OR success = 0"
df_fail = pd.read_sql_query(query_fail, conn)

print(f"전체 실패 기록 수: {len(df_fail)}개")
display(df_fail.head())

conn.close()

전체 실패 기록 수: 24941개


,timestamp,user_id,ip_address,location,device_type,os_type,browser,success,session_duration,target
0,2025-07-23 0:10,6a214818-e78e-47fc-a728-226c70b18e37,148.6.34.231,"Lake Robertbury, Bhutan",Tablet,Windows,Edge,FALSE,3167,TRUE
1,2026-01-16 7:21,a84f1f5d-ebde-4eb6-b5b2-2a3fe2976a4c,63.76.41.139,"New Deanna, Uzbekistan",Mobile,Android,Safari,FALSE,40,FALSE
2,2025-11-10 23:46,fad8810f-9df1-47fb-b2dc-d9ec37d565e7,97.73.84.28,"South Hannahmouth, Israel",PC,Linux,Opera,FALSE,788,TRUE
3,2025-07-31 7:27,bc5bebc1-4a8e-4353-a002-512cc92cd503,208.102.7.22,"Elizabethbury, Philippines",Mobile,iOS,Safari,FALSE,2508,TRUE
4,2025-05-31 3:16,9b430f47-4280-4c68-a55b-e989d1e4ac0e,61.5.6.68,"New Shawnastad, Greenland",Tablet,Android,Firefox,FALSE,840,FALSE


In [4]:
import torch
import torch.nn as nn

# 1. 딥러닝 모델(Autoencoder) 뼈대 설계
class LoginAnomalyDetector(nn.Module):
    def __init__(self, input_dim):
        super(LoginAnomalyDetector, self).__init__()
        
        # 인코더: 입력된 행렬을 압축 (특징 추출)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )
        
        # 디코더: 압축된 행렬을 다시 원래대로 복원
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [5]:
# 2. 위험 지수 및 가중치 계산 함수
def calculate_risk_score(reconstruction_error, is_overseas, is_new_device, fail_count, is_odd_time):
    # 기본 위험 지수는 모델의 복원 오차(Loss)를 기반으로 산출 (예: 오차 * 10)
    base_risk = reconstruction_error * 10
    
    # 의심 사유에 따른 가중치(Penalty) 계산[cite: 1]
    weight = 0
    if is_overseas:
        weight += 20  # 해외 IP 가중치[cite: 1]
    if is_new_device:
        weight += 15  # 새 기기 가중치[cite: 1]
    if is_odd_time:
        weight += 10  # 비정상적 시간대 가중치[cite: 1]
    
    # 실패 반복에 따른 가중치 (n회 이상 실패 시 기하급수적으로 증가)[cite: 1]
    if fail_count >= 3:
        weight += (fail_count * 5) 
        
    # 최종 위험 지수 산출 (최대 100%로 제한)
    total_risk_score = min(100, base_risk + weight)
    
    return total_risk_score

In [6]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

# (5줄짜리 df_preview를 임시로 사용)
df_dummy = df_preview.copy()

def preprocess_data(df):
    # 1. 딥러닝 모델에 넣기 위해 문자열 데이터들을 원-핫 인코딩 (0과 1의 벡터로 변환)
    # 제안서 기준 타겟 컬럼들 변환
    df_encoded = pd.get_dummies(df, columns=['device_type', 'os_type', 'browser'])
    
    # 2. 모델 연산에 당장 필요 없는 문자열(ID, IP 등)은 일단 제외하고 숫자 데이터만 남김
    # (나중에 위험 지수 계산 로직에서 따로 쓸 예정)
    numeric_df = df_encoded.select_dtypes(include=['float64', 'int64', 'bool'])
    
    # boolean(True/False) 값들을 실수형(1.0, 0.0) 행렬로 변환
    numeric_df = numeric_df.astype(float)
    
    # 3. Pandas 표 형태를 선형대수 연산을 위한 PyTorch Tensor(행렬)로 최종 변환
    tensor_data = torch.tensor(numeric_df.values, dtype=torch.float32)
    return tensor_data

# 전처리 함수 실행
tensor_data = preprocess_data(df_dummy)
input_dimension = tensor_data.shape[1] # 입력 벡터의 차원 수 계산

print(f"전처리 완료! 입력 벡터 차원 수(특성 개수): {input_dimension}개")
print(f"변환된 텐서의 형태(행렬 크기): {tensor_data.shape}")

전처리 완료! 입력 벡터 차원 수(특성 개수): 11개
변환된 텐서의 형태(행렬 크기): torch.Size([5, 11])


In [9]:
import torch.optim as optim

# 1. 모델, 손실 함수, 최적화 기법 세팅
# 아까 설계한 뼈대 모델을 불러오고, 입력 차원 수를 딱 맞게 꽂아줌
model = LoginAnomalyDetector(input_dim=input_dimension) 

# 오차를 계산하는 함수 (평균 제곱 오차)
criterion = nn.MSELoss() 
# 가중치 행렬을 미분(Gradient)을 통해 업데이트해 주는 최적화 도구 (Adam)
optimizer = optim.Adam(model.parameters(), lr=0.001) 

# 데이터를 조금씩 쪼개서 모델에 먹여주기 위한 DataLoader 설정
dataset = TensorDataset(tensor_data)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# 2. 본격적인 학습 루프 시작 (Epoch)
epochs = 5 # 임시로 5번만 반복 학습
print("🚀 모델 훈련 시작...")

for epoch in range(epochs):
    total_loss = 0
    for batch_data in dataloader:
        x = batch_data[0] # 입력 벡터
        
        # 순전파(Forward): 모델에 데이터 넣고 복원값 예측
        reconstructed_x = model(x)
        
        # 오차(Loss) 계산
        loss = criterion(reconstructed_x, x)
        
        # 역전파(Backpropagation): 오차를 바탕으로 미분하여 가중치 업데이트
        optimizer.zero_grad() # 이전 기울기 초기화
        loss.backward()       # 기울기 계산
        optimizer.step()      # 가중치 갱신
        
        total_loss += loss.item()
        
    # 각 에포크마다 평균 오차 출력 (숫자가 점점 줄어들면 학습이 잘 되고 있는 것!)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss(오차): {total_loss/len(dataloader):.4f}")

print("정상성 판단 모델 학습 완료")

🚀 모델 훈련 시작...
Epoch [1/5] - Loss(오차): 422114.8890
Epoch [2/5] - Loss(오차): 583025.6354
Epoch [3/5] - Loss(오차): 477664.9271
Epoch [4/5] - Loss(오차): 559306.1146
Epoch [5/5] - Loss(오차): 574120.4792
정상성 판단 모델 학습 완료


In [18]:
!pip install solapi

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 14.5 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)

   -------- ------------------------------- 2/9 [h11]
   ----------------- ---------------------- 4/9 [pydantic]
   -----------

In [ ]:
#전송 확인용 테스트 코드

from solapi import SolapiMessageService
from solapi.model import RequestMessage

API_KEY = "API_KEY 입력"
API_SECRET = "API_SECRET 입력"

MY_NUMBER = "01080612013" 

# 솔라피 서버 연결
message_service = SolapiMessageService(api_key=API_KEY, api_secret=API_SECRET)

# 문자 내용
message = RequestMessage(
    from_=MY_NUMBER,
    to=MY_NUMBER,
    text="[PIG] 테스트용으로 보내졌습니다."
)

try:
    response = message_service.send(message)
    print("전송 성공")
except Exception as e:
    print(f"에러 발생: {e}")

✅ 전송 성공! 당장 핸드폰 울렸는지 확인해 봐!


In [ ]:
#위험지수계산 및 알림전송

from solapi import SolapiMessageService
from solapi.model import RequestMessage

# 1. 솔라피 세팅
API_KEY = "여기에_발급받은_API_KEY_입력"
API_SECRET = "여기에_발급받은_API_SECRET_입력"
MY_NUMBER = "01012345678"

message_service = SolapiMessageService(api_key=API_KEY, api_secret=API_SECRET)

# 임시 DB (로그인 기록 저장용)
risk_score_db = []

# 2. 마스터 함수 (계산부터 알림, 저장까지 한 번에!)
def process_login_and_alert(user_id, reconstruction_error, is_overseas, is_new_device, fail_count, is_odd_time):
    
    # 아까 만들어둔 함수로 위험 지수(%) 계산
    final_score = calculate_risk_score(reconstruction_error, is_overseas, is_new_device, fail_count, is_odd_time)
    
    print(f"📊 [{user_id}] 계정 해킹 위험 지수: {final_score}%")
    status = "정상"
    
    # 위험 지수가 70% 이상이면 즉시 솔라피 문자로 경고 발송!
    if final_score >= 70:
        status = "위험"
        print("🚨 위험 지수 70% 돌파! 긴급 경고 문자를 발송합니다...")
        
        sms_text = f"🚨 [PIG 보안 시스템] {user_id} 계정의 해킹 위험 지수가 {final_score}%입니다. 즉시 비밀번호를 변경해주세요!"
        
        try:
            message = RequestMessage(
                from_=MY_NUMBER, 
                to=MY_NUMBER, # 나중엔 여기를 데이터베이스의 '유저 실제 번호'로 바꾸면 됨!
                text=sms_text
            )
            message_service.send(message)
            print("경고 문자 전송 완료!")
        except Exception as e:
            print(f"문자 전송 실패: {e}")
            
    elif final_score >= 40:
        status = "의심"
        print("⚠️ [주의] 비정상적인 로그인 시도 감지")
    else:
        print("✅ 안전한 로그인입니다.")
        
    # 데이터베이스에 기록 저장
    login_record = {
        'user_id': user_id,
        'risk_score': final_score,
        'status': status
    }
    risk_score_db.append(login_record)
    
    return login_record

# --- 최종 테스트 실행 ---
print("=== 해커 침입 시뮬레이션 ===")
process_login_and_alert(user_id="hacker_999", reconstruction_error=4.5, is_overseas=True, is_new_device=True, fail_count=3, is_odd_time=True)

=== 해커 침입 시뮬레이션 ===
📊 [hacker_999] 계정 해킹 위험 지수: 100%
🚨 위험 지수 70% 돌파! 긴급 경고 문자를 발송합니다...
❌ 📱 문자 전송 실패: 'ascii' codec can't encode characters in position 19-21: ordinal not in range(128)


{'user_id': 'hacker_999', 'risk_score': 100, 'status': '위험'}

In [ ]:
def process_login_and_alert(user_id, reconstruction_error, is_overseas, is_new_device, fail_count, is_odd_time):
    
    # [기본 계산] 위험 지수 산출
    final_score = calculate_risk_score(reconstruction_error, is_overseas, is_new_device, fail_count, is_odd_time)
    status = "정상"
    
   
    # 비정상적 로그인 발생 시 알림 (AI 모델의 독립적 판단)

    # AI 모델의 오차값(reconstruction_error)이 기준치(예: 3.0)를 넘은 경우
    if reconstruction_error >= 3.0:
        status = "의심"
        print(f"⚠️ [알림 1] AI 모델 감지: {user_id} 계정에서 비정상적인 로그인 패턴이 포착되었습니다.")
        # (필요시 내부 보안 로그 저장 및 프론트엔드에 의심 신호 전송)

   
    # 위험지수 특정값 도달 시 알림 (최종 융합 점수 기준)

    # 가중치가 합산된 최종 위험 지수가 특정 값(예: 70%) 이상인 경우
    if final_score >= 70:
        status = "위험"
        print(f"🚨 [알림 2] 긴급: {user_id} 계정의 최종 위험 지수가 {final_score}%에 도달했습니다.")
        
        # 기기 차단 및 유저 핸드폰으로 솔라피 문자 강제 발송!
        sms_text = f"🚨 [보안 경고] {user_id} 계정의 해킹 위험 지수가 {final_score}%입니다. 본인이 아닐 경우 즉시 비밀번호를 변경하세요."
        try:
            message = RequestMessage(from_=MY_NUMBER, to=MY_NUMBER, text=sms_text)
            message_service.send(message)
            print("솔라피 긴급 경고 문자 전송 완료")
        except Exception as e:
            print(f"문자 전송 실패: {e}")
            
    
    # DB 기록 저장
    login_record = {
        'user_id': user_id,
        'ai_anomaly': "비정상" if reconstruction_error >= 3.0 else "정상",
        'risk_score': final_score,
        'status': status
    }
    risk_score_db.append(login_record)
    
    return login_record